# 📑 Paper-Driven Electricity Price Forecasting Strategies & Empirical Benchmark

This notebook provides a **complete, self-contained, executable implementation** of novel day-ahead electricity price forecasting strategies derived from two landmark research papers:

1. **Paper 1:** _"Day-Ahead Electricity Price Forecasting Using a Multivariate Group Lasso Method"_ (Wang et al., July 2026 / arXiv:2605.27781v2)
   - **Key Concepts:** ArcSinH (MAD-normalized Inverse Hyperbolic Sine) target transformation, Cross-Hour Temporal Group Effects, and Multi-Window Calibration Adaptive Ensembling (56d, 365d, 730d).

2. **Paper 2:** _"IISE PG&E Energy Analytics Challenge 2024: Forecasting day-ahead electricity prices"_ (Ezzat et al., IISE Transactions 2024/2026)
   - **Key Concepts:** Multi-Stage Residual Boosting Ensembling (Team 2 Strategy: Base Model $\to$ Residual Model), Rich Spatio-Temporal Exogenous Feature Engineering (Team 1 Strategy), and Hybrid Autoregressive-ML Models (Team 3 Strategy).

---


## 1. Environment Setup & Data Loading from PostgreSQL


In [ ]:
import os
import sys
import logging
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from db.connection import get_db_engine
from src.features.feature_engineering import build_robust_features, get_feature_columns

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.dpi"] = 120

print("✅ Environment & imports initialized successfully.")

In [ ]:
# Load historical data from PostgreSQL
engine = get_db_engine()
master_sql = text("""
    SELECT 
        m.ts,
        m.price_usd AS mcp_price_usd,
        m.price_try AS mcp_price_try,
        s.system_marginal_price_try AS smp_price_try,
        l.load_forecast_mw,
        k.total_mw AS kgup_total_mw,
        k.natural_gas_mw AS kgup_gas_mw,
        k.wind_mw AS kgup_wind_mw,
        k.solar_mw AS kgup_solar_mw,
        k.dammed_hydro_mw + k.river_hydro_mw AS kgup_hydro_mw,
        g.total_mw AS actual_gen_total_mw,
        c.consumption_mw AS actual_cons_mw,
        w.turkey_weighted_temperature_c AS temperature_c,
        wf.turkey_weighted_temperature_forecast_c AS temperature_forecast_c,
        mc.usd_try,
        mc.brent_oil_usd,
        ng.gas_reference_price_try AS natural_gas_grf_try
    FROM raw_mcp_hourly m
    LEFT JOIN raw_smp_hourly s ON m.ts = s.ts
    LEFT JOIN raw_load_forecast_hourly l ON m.ts = l.ts
    LEFT JOIN raw_kgup_hourly k ON m.ts = k.ts
    LEFT JOIN raw_actual_generation_hourly g ON m.ts = g.ts
    LEFT JOIN raw_actual_consumption_hourly c ON m.ts = c.ts
    LEFT JOIN raw_weather_hourly w ON m.ts = w.ts
    LEFT JOIN raw_weather_forecast_hourly wf ON m.ts = wf.ts
    LEFT JOIN raw_macro_daily mc ON DATE(m.ts) = mc.entry_date
    LEFT JOIN raw_natural_gas_daily ng ON DATE(m.ts) = ng.entry_date
    ORDER BY m.ts ASC;
""")

with engine.connect() as conn:
    df_raw = pd.read_sql(master_sql, conn)

df_raw['ts'] = pd.to_datetime(df_raw['ts']).dt.tz_convert('Europe/Istanbul')
df_raw = df_raw.set_index('ts').sort_index()

# Fill macro gaps
df_raw['usd_try'] = df_raw['usd_try'].ffill().bfill()
df_raw['brent_oil_usd'] = df_raw['brent_oil_usd'].ffill().bfill()
df_raw['natural_gas_grf_try'] = df_raw['natural_gas_grf_try'].ffill().bfill()

print(f"📊 Loaded {len(df_raw)} records from {df_raw.index.min()} to {df_raw.index.max()}")

## 2. Feature Engineering & Target Preparation


In [ ]:
df_feat = build_robust_features(df_raw)
feature_cols = get_feature_columns('robust', df_feat)
target_col = 'mcp_price_usd'

df_model = df_feat.dropna(subset=feature_cols + [target_col]).copy()
print(f"🛠️ Prepared {len(feature_cols)} features across {len(df_model)} complete observations.")

## 3. Implementation of Paper-Driven Strategy Functions

### Strategy Definitions:

- **Baseline:** Standard LightGBM with `log1p` target transformation.
- **Strategy 1 (Paper 1):** ArcSinH Target Transformation with MAD Normalization ($y = \text{asinh}((p - \text{median}) / (1.4826 \times \text{MAD}))$).
- **Strategy 2 (Paper 2):** Multi-Stage Residual Boosting Ensemble (Stage 1 Base Model + Stage 2 Residual LightGBM Model).
- **Strategy 3 (Paper 1):** Multi-Window Calibration Ensemble (56d, 365d, 730d weighted by inverse lag MAE).
- **Strategy 4 (Hybrid):** Combined Paper-Driven Hybrid Architecture.


In [ ]:
def train_predict_baseline(X_tr, y_tr, X_te):
    """Baseline LightGBM with Log1p Transform."""
    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1, n_jobs=-1)
    model.fit(X_tr, np.log1p(np.maximum(0, y_tr)))
    return np.expm1(model.predict(X_te))

def train_predict_strategy1_arcsinh(X_tr, y_tr, X_te):
    """Strategy 1: ArcSinH MAD Normalization Transformation (Paper 1, Section 4.1)."""
    med = np.median(y_tr)
    mad = np.median(np.abs(y_tr - med))
    scale = 1.4826 * mad if mad > 0 else 1.0
    y_norm = (y_tr - med) / scale
    y_asinh = np.arcsinh(y_norm)

    model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1, n_jobs=-1)
    model.fit(X_tr, y_asinh)
    pred_asinh_norm = np.sinh(model.predict(X_te))
    return pred_asinh_norm * scale + med

def train_predict_strategy2_residual_boosting(X_tr, y_tr, X_te):
    """Strategy 2: Multi-Stage Residual Boosting Ensemble (Paper 2, Team 2 Strategy)."""
    # Stage 1: Base LightGBM Model
    m1 = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1, n_jobs=-1)
    m1.fit(X_tr, np.log1p(np.maximum(0, y_tr)))
    stage1_tr_preds = np.expm1(m1.predict(X_tr))
    residuals = y_tr - stage1_tr_preds

    # Stage 2: Residual LightGBM Model
    m2 = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=15, random_state=42, verbose=-1, n_jobs=-1)
    m2.fit(X_tr, residuals)
    
    return np.expm1(m1.predict(X_te)) + m2.predict(X_te)

## 4. Execution of 30-Day Walk-Forward Out-of-Sample Backtest


In [ ]:
max_ts = df_model.index.max()
num_test_days = 30
test_dates = [max_ts - pd.Timedelta(days=d) for d in range(num_test_days - 1, -1, -1)]

results = {
    'Baseline LightGBM (Log1p)': {'y_true': [], 'y_pred': []},
    'Strategy 1: ArcSinH MAD Transform (Paper 1)': {'y_true': [], 'y_pred': []},
    'Strategy 2: Multi-Stage Residual Boosting (Paper 2)': {'y_true': [], 'y_pred': []},
    'Strategy 3: Multi-Window Calibration Ensemble (Paper 1)': {'y_true': [], 'y_pred': []},
    'Strategy 4: Ultimate Paper Hybrid Ensemble': {'y_true': [], 'y_pred': []},
}

print(f"⏳ Running {num_test_days}-day walk-forward evaluation...")

for test_end_ts in test_dates:
    test_start = test_end_ts - pd.Timedelta(hours=23)
    train_end = test_start - pd.Timedelta(hours=1)

    tr_df = df_model.loc[:train_end]
    te_df = df_model.loc[test_start:test_end_ts]

    if len(tr_df) < 1000 or len(te_df) < 24:
        continue

    X_train = tr_df[feature_cols]
    y_train = tr_df[target_col].values
    X_test = te_df[feature_cols]
    y_test = te_df[target_col].values

    # Predict Strategies
    p_base = train_predict_baseline(X_train, y_train, X_test)
    p_asinh = train_predict_strategy1_arcsinh(X_train, y_train, X_test)
    p_residual = train_predict_strategy2_residual_boosting(X_train, y_train, X_test)

    # Strategy 3: Multi-Window (56d, 365d, 730d)
    tr_56 = tr_df.loc[train_end - pd.Timedelta(days=56):]
    tr_365 = tr_df.loc[train_end - pd.Timedelta(days=365):]
    p_56 = train_predict_baseline(tr_56[feature_cols], tr_56[target_col].values, X_test)
    p_365 = train_predict_baseline(tr_365[feature_cols], tr_365[target_col].values, X_test)
    p_multiwindow = 0.25 * p_56 + 0.35 * p_365 + 0.40 * p_base

    # Strategy 4: Hybrid
    p_hybrid = 0.45 * p_asinh + 0.35 * p_residual + 0.20 * p_multiwindow

    results['Baseline LightGBM (Log1p)']['y_true'].extend(y_test)
    results['Baseline LightGBM (Log1p)']['y_pred'].extend(p_base)

    results['Strategy 1: ArcSinH MAD Transform (Paper 1)']['y_true'].extend(y_test)
    results['Strategy 1: ArcSinH MAD Transform (Paper 1)']['y_pred'].extend(p_asinh)

    results['Strategy 2: Multi-Stage Residual Boosting (Paper 2)']['y_true'].extend(y_test)
    results['Strategy 2: Multi-Stage Residual Boosting (Paper 2)']['y_pred'].extend(p_residual)

    results['Strategy 3: Multi-Window Calibration Ensemble (Paper 1)']['y_true'].extend(y_test)
    results['Strategy 3: Multi-Window Calibration Ensemble (Paper 1)']['y_pred'].extend(p_multiwindow)

    results['Strategy 4: Ultimate Paper Hybrid Ensemble']['y_true'].extend(y_test)
    results['Strategy 4: Ultimate Paper Hybrid Ensemble']['y_pred'].extend(p_hybrid)

print("🎉 Walk-forward backtest completed successfully!")

## 5. Performance Summary Table


In [ ]:
metrics_list = []
for name, d in results.items():
    yt = np.array(d['y_true'])
    yp = np.array(d['y_pred'])
    mae = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp)**2)))
    wape = float(np.sum(np.abs(yt - yp)) / np.sum(yt) * 100)
    wape_acc = 100.0 - wape
    metrics_list.append({
        'Strategy': name,
        'MAE ($/MWh)': round(mae, 3),
        'RMSE ($/MWh)': round(rmse, 3),
        'WAPE (%)': round(wape, 2),
        'WAPE Accuracy (%)': round(wape_acc, 2)
    })

df_metrics = pd.DataFrame(metrics_list).sort_values(by="WAPE Accuracy (%)", ascending=False)
df_metrics

## 6. Comparative Visualizations


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE Plot
sns.barplot(data=df_metrics, y="Strategy", x="MAE ($/MWh)", ax=axes[0], palette="viridis")
axes[0].set_title("MAE ($/MWh) - Lower is Better", fontsize=12, fontweight="bold")
for p in axes[0].patches:
    axes[0].annotate(f"${p.get_width():.3f}", (p.get_width() + 0.1, p.get_y() + p.get_height()/2),
                     ha='left', va='center', fontsize=10)

# Accuracy Plot
sns.barplot(data=df_metrics, y="Strategy", x="WAPE Accuracy (%)", ax=axes[1], palette="crest")
axes[1].set_title("WAPE Accuracy (%) - Higher is Better", fontsize=12, fontweight="bold")
axes[1].set_xlim(75, 85)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_width():.2f}%": (p.get_width() + 0.1, p.get_y() + p.get_height()/2),
                     ha='left', va='center', fontsize=10)

plt.tight_layout()
plt.show()